# 🔧 Notebook 02 — Preprocessing Data Time Series
## Early Warning System Krisis Pariwisata Bali

Notebook ini memproses:
1. Wisman Gabungan (sudah bersih)
2. Wisatawan Domestik Bulanan (format wide BPS)
3. Kurs USD/IDR
4. Daily Forex Rates
5. TPK Hotel Bintang (format wide BPS)
6. Lama Menginap Hotel Bintang (format wide BPS)
7. Inflasi Bulanan Bali

**Output:** File CSV bersih di folder `data/processed/`

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✅ Library siap')

✅ Library siap


---
## 1. Wisman Gabungan — Gab_Data_Wisman_Bali.xlsx
Dataset ini sudah dalam format bersih (Tanggal, Banyak), hanya perlu rename dan convert.

In [6]:
# Load
wisman = pd.read_excel('data/raw/Gab_Data_Wisman_Bali.xlsx')

# Rename kolom
wisman.columns = ['date', 'wisman']

# Convert datetime
wisman['date'] = pd.to_datetime(wisman['date'])

# Sort ascending
wisman = wisman.sort_values('date').reset_index(drop=True)

# Buat kolom bulan (period)
wisman['month'] = wisman['date'].dt.to_period('M')

# Cek missing value
print('Missing values:', wisman.isnull().sum().to_dict())
print('Rentang data:', wisman['date'].min(), 'sampai', wisman['date'].max())
print('Total baris:', len(wisman))
print()
wisman.head(5)

Missing values: {'date': 0, 'wisman': 0, 'month': 0}
Rentang data: 2009-01-01 00:00:00 sampai 2024-12-01 00:00:00
Total baris: 192



,date,wisman,month
0,2009-01-01,174541,2009-01
1,2009-02-01,147704,2009-02
2,2009-03-01,168205,2009-03
3,2009-04-01,188776,2009-04
4,2009-05-01,190803,2009-05


In [7]:
# Simpan
wisman.to_csv('data/processed/wisman_clean.csv', index=False)
print('✅ wisman_clean.csv disimpan')

✅ wisman_clean.csv disimpan


---
## 2. Wisatawan Domestik Bulanan 2004-2025
Format BPS: wide table, baris=bulan, kolom=tahun. Perlu di-melt menjadi format panjang.

In [8]:
# Load raw — header null dulu
wisnus_raw = pd.read_excel(
    'data/raw/banyaknya-wisatawan-domestik-bulanan-ke-bali.xls',
    engine='xlrd',
    header=None
)

# Baris ke-3 adalah header (Bulan + tahun-tahun)
# Baris ke-4 ke bawah adalah data
header_row = wisnus_raw.iloc[3].tolist()
data_rows = wisnus_raw.iloc[4:].copy()
data_rows.columns = header_row
data_rows = data_rows.reset_index(drop=True)

print('Header:', header_row[:5], '...')
print('Shape data:', data_rows.shape)
data_rows.head(5)

Header: ['Bulan', 2004, np.float64(2005.0), np.float64(2006.0), np.float64(2007.0)] ...
Shape data: (49, 10)


,Bulan,2004,2005.0,2006.0,2007.0,2008.0,2009.0,2010.0,2011.0,2012.0
0,Januari,167106,174515.0,202857.0,181266.0,225955.0,264915.0,349575.0,280588.0,333199.0
1,Pebruari,133660,161808.0,161413.0,144425.0,190792.0,204419.0,238789.0,340508.0,305934.0
2,Maret,118369,194411.0,171795.0,161009.0,221181.0,255203.0,202995.0,358313.0,307616.0
3,April,129730,174033.0,192182.0,165509.0,206631.0,247100.0,396898.0,385228.0,331378.0
4,Mei,142186,190855.0,188152.0,183736.0,226339.0,289635.0,421369.0,463452.0,525076.0


In [9]:
# Identifikasi kolom bulan dan kolom tahun
bulan_col = data_rows.columns[0]  # kolom pertama = nama bulan
tahun_cols = [c for c in data_rows.columns[1:] if str(c) != 'nan' and str(c) != 'None']

# Filter baris yang valid (12 bulan)
bulan_names = ['Januari', 'Pebruari', 'Februari', 'Maret', 'April', 'Mei',
               'Juni', 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember']
data_rows = data_rows[data_rows[bulan_col].astype(str).str.strip().isin(bulan_names)].copy()

# Normalisasi nama bulan
data_rows[bulan_col] = data_rows[bulan_col].str.strip().replace({'Pebruari': 'Februari'})

print('Bulan yang tersedia:', data_rows[bulan_col].unique().tolist())
print('Tahun yang tersedia (sample):', tahun_cols[:8])

Bulan yang tersedia: ['Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 'Juli', 'Agustus', 'September', 'Oktober', 'Desember']
Tahun yang tersedia (sample): [2004, np.float64(2005.0), np.float64(2006.0), np.float64(2007.0), np.float64(2008.0), np.float64(2009.0), np.float64(2010.0), np.float64(2011.0)]


In [10]:
# Melt: wide -> long format
wisnus_long = data_rows.melt(
    id_vars=[bulan_col],
    value_vars=tahun_cols,
    var_name='tahun',
    value_name='wisnus'
)

# Buat kolom tanggal dari bulan + tahun
bulan_map = {
    'Januari': '01', 'Februari': '02', 'Maret': '03', 'April': '04',
    'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
    'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
}
wisnus_long['bulan_num'] = wisnus_long[bulan_col].map(bulan_map)
wisnus_long['tahun'] = wisnus_long['tahun'].astype(str).str[:4]  # ambil 4 digit
wisnus_long['date'] = pd.to_datetime(
    wisnus_long['tahun'] + '-' + wisnus_long['bulan_num'] + '-01',
    errors='coerce'
)

# Bersihkan
wisnus_long['wisnus'] = pd.to_numeric(wisnus_long['wisnus'], errors='coerce')
wisnus_long = wisnus_long[['date', 'wisnus']].dropna()
wisnus_long = wisnus_long.sort_values('date').reset_index(drop=True)
wisnus_long['month'] = wisnus_long['date'].dt.to_period('M')

print('Shape:', wisnus_long.shape)
print('Rentang:', wisnus_long['date'].min(), '->', wisnus_long['date'].max())
wisnus_long.head(5)

Shape: (242, 3)
Rentang: 2004-01-01 00:00:00 -> 2012-12-01 00:00:00


,date,wisnus,month
0,2004-01-01,167106.0,2004-01
1,2004-01-01,426360.0,2004-01
2,2004-01-01,527447.0,2004-01
3,2004-02-01,133660.0,2004-02
4,2004-02-01,369525.0,2004-02


In [11]:
wisnus_long.to_csv('data/processed/wisnus_clean.csv', index=False)
print('✅ wisnus_clean.csv disimpan')

✅ wisnus_clean.csv disimpan


---
## 3. Kurs USD/IDR Historical
Format: Date | Price | Open | High | Low | Change %

In [13]:
# Load
usd = pd.read_csv('data/raw/USD_IDR Historical Data.csv')

# Ambil kolom yang dibutuhkan
usd = usd[['Date', 'Price']].copy()
usd.columns = ['date', 'usd_idr']

# Convert date
usd['date'] = pd.to_datetime(usd['date'], format='%m/%d/%Y')

# Hapus koma dari harga (format: 16,090.00)
usd['usd_idr'] = usd['usd_idr'].astype(str).str.replace(',', '')
usd['usd_idr'] = pd.to_numeric(usd['usd_idr'], errors='coerce')

# Sort ascending
usd = usd.sort_values('date').reset_index(drop=True)

# Agregasi bulanan (rata-rata kurs per bulan)
usd['month'] = usd['date'].dt.to_period('M')
monthly_usd = usd.groupby('month')['usd_idr'].mean().reset_index()
monthly_usd.columns = ['month', 'usd_idr_avg']

print('Shape harian:', usd.shape)
print('Shape bulanan:', monthly_usd.shape)
print('Rentang:', usd['date'].min(), '->', usd['date'].max())
print()
monthly_usd.tail(5)

Shape harian: (3858, 3)
Shape bulanan: (180, 2)
Rentang: 2010-01-01 00:00:00 -> 2024-12-31 00:00:00



,month,usd_idr_avg
175,2024-08,15734.772727
176,2024-09,15318.480952
177,2024-10,15557.608696
178,2024-11,15813.095238
179,2024-12,16035.681818


In [14]:
monthly_usd.to_csv('data/processed/monthly_usd.csv', index=False)
print('✅ monthly_usd.csv disimpan')

✅ monthly_usd.csv disimpan


---
## 4. Daily Forex Rates (Filter IDR)

In [15]:
# Load
forex = pd.read_csv('data/raw/daily_forex_rates.csv')

# Filter hanya IDR
forex_idr = forex[forex['currency'] == 'IDR'].copy()
forex_idr = forex_idr[['date', 'exchange_rate']].copy()

# Convert date
forex_idr['date'] = pd.to_datetime(forex_idr['date'])
forex_idr = forex_idr.sort_values('date').reset_index(drop=True)

# Agregasi bulanan
forex_idr['month'] = forex_idr['date'].dt.to_period('M')
monthly_forex = forex_idr.groupby('month')['exchange_rate'].mean().reset_index()
monthly_forex.columns = ['month', 'idr_eur_rate']

print('Shape harian:', forex_idr.shape)
print('Shape bulanan:', monthly_forex.shape)
print()
monthly_forex.tail(5)

Shape harian: (3263, 3)
Shape bulanan: (139, 2)



,month,idr_eur_rate
134,2026-01,19729.961342
135,2026-02,19903.853782
136,2026-03,19595.017187
137,2026-04,20026.122880
138,2026-05,20459.534496


In [16]:
monthly_forex.to_csv('data/processed/monthly_forex.csv', index=False)
print('✅ monthly_forex.csv disimpan')

✅ monthly_forex.csv disimpan


---
## 5. TPK Hotel Bintang
Format BPS: baris=kelas bintang, kolom=bulan, setiap sheet=satu tahun.
Kita ambil rata-rata semua kelas bintang per bulan.

In [17]:
# Load raw
tpk_raw = pd.read_excel(
    'data/raw/Tingkat Penghunian Kamar (TPK) Hotel Bintang.xlsx',
    header=None
)

# Deteksi tahun dari baris ke-2
tahun_cell = str(tpk_raw.iloc[2, 1]).strip()
print('Tahun terdeteksi:', tahun_cell)

# Bulan di baris ke-3
bulan_headers = tpk_raw.iloc[3, 1:13].tolist()
print('Bulan:', bulan_headers)

# Data mulai baris ke-4
data = tpk_raw.iloc[4:].copy()
data.columns = ['kelas'] + bulan_headers + ['tahunan'] + list(range(len(data.columns)-14))

# Filter kelas bintang yang valid
kelas_valid = ['Bintang 5', 'Bintang 4', 'Bintang 3', 'Bintang 2', 'Bintang 1', 'Jumlah / Total']
data = data[data['kelas'].astype(str).str.strip().isin(kelas_valid)].copy()

print('Kelas ditemukan:', data['kelas'].tolist())
data[['kelas'] + bulan_headers[:6]]

Tahun terdeteksi: 2000
Bulan: ['Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember']
Kelas ditemukan: ['Bintang 5', 'Bintang 4', 'Bintang 3', 'Bintang 2', 'Bintang 1']


,kelas,Januari,Februari,Maret,April,Mei,Juni
4,Bintang 5,56.5,59.34,58.49,62.19,61.07,68.6
5,Bintang 4,53.11,47.51,49.13,48.77,46.03,52.97
6,Bintang 3,44.41,43.55,39.68,48.55,41.2,50.48
7,Bintang 2,43.87,42.66,40.68,44.59,39.87,43.91
8,Bintang 1,37.83,43.77,36.77,34.71,23.7,27.84


In [18]:
# Ambil baris 'Jumlah / Total' atau rata-rata semua kelas
bulan_map = {
    'Januari': '01', 'Pebruari': '02', 'Februari': '02', 'Maret': '03',
    'April': '04', 'Mei': '05', 'Juni': '06', 'Juli': '07',
    'Agustus': '08', 'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
}

# Hitung rata-rata TPK semua kelas per bulan
tpk_records = []
for bulan in bulan_headers:
    if bulan in bulan_map:
        bulan_num = bulan_map[bulan]
        values = pd.to_numeric(data[bulan], errors='coerce').dropna()
        if len(values) > 0:
            avg_tpk = values.mean()
            date_str = f"{tahun_cell}-{bulan_num}-01"
            tpk_records.append({'month_str': date_str, 'tpk_avg': avg_tpk})

tpk_df = pd.DataFrame(tpk_records)
tpk_df['date'] = pd.to_datetime(tpk_df['month_str'])
tpk_df['month'] = tpk_df['date'].dt.to_period('M')
tpk_df = tpk_df[['month', 'date', 'tpk_avg']]

print('Shape TPK:', tpk_df.shape)
print('⚠️  CATATAN: File ini hanya berisi 1 tahun. Untuk multi-tahun, ulangi proses per file.')
tpk_df

Shape TPK: (12, 3)
⚠️  CATATAN: File ini hanya berisi 1 tahun. Untuk multi-tahun, ulangi proses per file.


,month,date,tpk_avg
0,2000-01,2000-01-01,47.144
1,2000-02,2000-02-01,47.366
2,2000-03,2000-03-01,44.950
3,2000-04,2000-04-01,47.762
4,2000-05,2000-05-01,42.374
5,2000-06,2000-06-01,48.760
6,2000-07,2000-07-01,57.218
7,2000-08,2000-08-01,55.064
8,2000-09,2000-09-01,55.484
9,2000-10,2000-10-01,53.694


In [19]:
tpk_df.to_csv('data/processed/tpk_clean.csv', index=False)
print('✅ tpk_clean.csv disimpan')

✅ tpk_clean.csv disimpan


---
## 6. Inflasi Bulanan Bali 2024
Format: baris=kota, kolom=bulan. Ambil baris Provinsi Bali atau rata-rata.

In [20]:
# Load raw
inflasi_raw = pd.read_excel('data/raw/Inflasi_Bulanan.xlsx', header=None)

# Baris ke-2: tahun, baris ke-3: nama bulan, data mulai baris ke-4
tahun_inflasi = str(inflasi_raw.iloc[2, 1]).strip()
bulan_inflasi = inflasi_raw.iloc[3, 1:13].tolist()
print('Tahun:', tahun_inflasi)
print('Bulan:', bulan_inflasi)

data_inflasi = inflasi_raw.iloc[4:].copy()
data_inflasi.columns = ['kota'] + bulan_inflasi + ['tahunan'] + list(range(len(data_inflasi.columns)-14))

# Filter baris yang valid (ada nama kotanya)
data_inflasi = data_inflasi[data_inflasi['kota'].notna()].copy()
data_inflasi['kota'] = data_inflasi['kota'].astype(str).str.strip()
data_inflasi = data_inflasi[data_inflasi['kota'] != 'nan'].copy()

print('Kota tersedia:', data_inflasi['kota'].tolist())
data_inflasi[['kota'] + bulan_inflasi]

Tahun: 2024
Bulan: ['Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember']
Kota tersedia: ['Kabupaten Tabanan', 'Kabupaten Badung', 'Singaraja', 'Kota Denpasar', 'Provinsi Bali', 'Nasional']


,kota,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember
4,Kabupaten Tabanan,-0.07,0.68,0.91,0.42,-0.28,-1.09,0.09,0.28,0.26,-0.03,0.76,0.49
5,Kabupaten Badung,-0.01,0.58,1.1,0.03,-0.09,-0.63,-0.03,-0.09,0.09,-0.02,0.68,0.37
6,Singaraja,-0.22,0.51,0.89,0.07,-0.33,-0.53,0.12,-0.18,0.25,0.21,0.81,0.32
7,Kota Denpasar,-0.08,0.65,0.87,0.53,0.05,-0.32,0.16,0.26,0.06,0.1,0.19,0.19
8,Provinsi Bali,-0.09,0.61,0.93,0.32,-0.1,-0.55,0.1,0.1,0.13,0.07,0.5,0.31
9,Nasional,0.04,0.37,0.52,0.25,-0.03,-0.08,-0.18,-0.03,-0.12,0.08,0.3,0.44


In [21]:
# Hitung rata-rata inflasi semua kota per bulan
bulan_map = {
    'Januari': '01', 'Februari': '02', 'Pebruari': '02', 'Maret': '03',
    'April': '04', 'Mei': '05', 'Juni': '06', 'Juli': '07',
    'Agustus': '08', 'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
}

inflasi_records = []
for bulan in bulan_inflasi:
    if bulan in bulan_map:
        values = pd.to_numeric(data_inflasi[bulan], errors='coerce').dropna()
        if len(values) > 0:
            inflasi_records.append({
                'date': f"{tahun_inflasi}-{bulan_map[bulan]}-01",
                'inflasi_bali': values.mean()
            })

inflasi_df = pd.DataFrame(inflasi_records)
inflasi_df['date'] = pd.to_datetime(inflasi_df['date'])
inflasi_df['month'] = inflasi_df['date'].dt.to_period('M')

print('Shape:', inflasi_df.shape)
print('⚠️  Catatan: Data inflasi hanya tersedia untuk 1 tahun di file ini.')
inflasi_df

Shape: (12, 3)
⚠️  Catatan: Data inflasi hanya tersedia untuk 1 tahun di file ini.


,date,inflasi_bali,month
0,2024-01-01,-0.071667,2024-01
1,2024-02-01,0.566667,2024-02
2,2024-03-01,0.870000,2024-03
3,2024-04-01,0.270000,2024-04
4,2024-05-01,-0.130000,2024-05
5,2024-06-01,-0.533333,2024-06
6,2024-07-01,0.043333,2024-07
7,2024-08-01,0.056667,2024-08
8,2024-09-01,0.111667,2024-09
9,2024-10-01,0.068333,2024-10


In [22]:
inflasi_df.to_csv('data/processed/inflasi_clean.csv', index=False)
print('✅ inflasi_clean.csv disimpan')

✅ inflasi_clean.csv disimpan


---
## 7. Wisman Bali vs Indonesia Tahunan (1969–2025)
Format: dua tabel side-by-side dalam satu sheet. Baris header di row ke-4/5, data mulai row ke-6.
Berguna untuk menghitung **bali_share_pct** (porsi wisman Bali dari total wisman Indonesia).


In [23]:
# Load raw
wisman_indo_raw = pd.read_excel(
    'data/raw/banyaknya-wisatawan-mancanegara-ke-bali-dan-indonesia.xls',
    engine='xlrd',
    header=None
)

# Tabel split jadi dua bagian: kolom 0-4 (kiri) dan kolom 6-10 (kanan)
# Baris 3: header tahun/total, baris 4: sub-header (Total, Growth%)
# Data mulai baris ke-5

def parse_half_table(df, col_start, col_end):
    """Parse satu sisi tabel wisman Bali vs Indonesia."""
    sub = df.iloc[5:, col_start:col_end+1].copy()
    sub.columns = ['tahun', 'indonesia_total', 'indonesia_growth', 'bali_total', 'bali_growth']
    sub = sub.dropna(subset=['tahun'])
    sub['tahun'] = pd.to_numeric(sub['tahun'], errors='coerce')
    sub = sub.dropna(subset=['tahun'])
    sub['tahun'] = sub['tahun'].astype(int)
    sub['indonesia_total'] = pd.to_numeric(sub['indonesia_total'], errors='coerce')
    sub['bali_total'] = pd.to_numeric(sub['bali_total'], errors='coerce')
    return sub[['tahun', 'indonesia_total', 'bali_total']]

# Parse kedua sisi lalu gabung
left  = parse_half_table(wisman_indo_raw, 0, 4)
right = parse_half_table(wisman_indo_raw, 6, 10)
wisman_indo_annual = pd.concat([left, right], ignore_index=True)
wisman_indo_annual = wisman_indo_annual.sort_values('tahun').reset_index(drop=True)

# Hitung bali_share_pct: porsi wisman Bali dari total Indonesia
wisman_indo_annual['bali_share_pct'] = (
    wisman_indo_annual['bali_total'] / wisman_indo_annual['indonesia_total'] * 100
).round(2)

print('Shape:', wisman_indo_annual.shape)
print('Rentang tahun:', wisman_indo_annual['tahun'].min(), '-', wisman_indo_annual['tahun'].max())
print()
wisman_indo_annual.tail(10)


Shape: (57, 4)
Rentang tahun: 1969 - 2025



,tahun,indonesia_total,bali_total,bali_share_pct
47,2016,11519275,4927937,42.78
48,2017,14039799,5697739,40.58
49,2018,15806191,6070473,38.41
50,2019,16106954,6275210,38.96
51,2020,4052923,1069473,26.39
52,2021,1557530,51,0.00
53,2022,5889031,2155747,36.61
54,2023,11677825,5273258,45.16
55,2024,13902420,6333360,45.56
56,2025,15386646,6948754,45.16


In [24]:
wisman_indo_annual.to_csv('data/processed/wisman_vs_indonesia_clean.csv', index=False)
print('✅ wisman_vs_indonesia_clean.csv disimpan')
print('Kolom:', wisman_indo_annual.columns.tolist())
print('Catatan: Data tahunan — akan di-join ke timeline bulanan berdasarkan tahun di NB04')


✅ wisman_vs_indonesia_clean.csv disimpan
Kolom: ['tahun', 'indonesia_total', 'bali_total', 'bali_share_pct']
Catatan: Data tahunan — akan di-join ke timeline bulanan berdasarkan tahun di NB04


---
## 8. Cek Semua Output

In [25]:
import os

processed_files = os.listdir('data/processed/')
print('File di data/processed/:')
for f in sorted(processed_files):
    if f.endswith('.csv'):
        df = pd.read_csv(f'data/processed/{f}')
        print(f'  ✅ {f} — {df.shape[0]} baris, {df.shape[1]} kolom')

File di data/processed/:
  ✅ inflasi_clean.csv — 12 baris, 3 kolom
  ✅ monthly_forex.csv — 139 baris, 2 kolom
  ✅ monthly_usd.csv — 180 baris, 2 kolom
  ✅ tpk_clean.csv — 12 baris, 3 kolom
  ✅ wisman_clean.csv — 192 baris, 3 kolom
  ✅ wisman_vs_indonesia_clean.csv — 57 baris, 4 kolom
  ✅ wisnus_clean.csv — 242 baris, 3 kolom
